In [0]:
import logging
import yaml
import requests
from tenacity import retry, stop_after_attempt, wait_exponential

logger = logging.getLogger("eia_ingestion")
logger.setLevel(logging.INFO)

dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("config_path", "")

environment = dbutils.widgets.get("environment")
CONFIG_PATH = dbutils.widgets.get("config_path")

try:
    with open(CONFIG_PATH) as f:
        full_config = yaml.safe_load(f)

    config = full_config[environment]
    volume_path = config["bronze_volumes_path"]

    logger.info(f"[{environment}] Config loaded. Volume path: {volume_path}")

except Exception as e:
    logger.error(f"Failed to load config from {CONFIG_PATH} for environment '{environment}': {e}")
    raise

API_KEY = dbutils.secrets.get(scope="eia_api", key="eia-api-key")

In [0]:
from datetime import date, timedelta

CONSUMPTION_URL = "https://api.eia.gov/v2/petroleum/cons/wpsup/data/"
PRICES_URL = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=8))
def fetch_page(url, offset, start, end, length=5000, extra_params=None):
    params = {
        "frequency": "weekly",
        "data[0]": "value",
        "start": start,
        "end": end,
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "offset": offset,
        "length": length,
        "api_key": API_KEY,
    }
    if extra_params:
        params.update(extra_params)
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response


def fetch_all(url, start, end, extra_params=None, page_size=5000):
    all_data = []
    offset = 0
    total = None

    while total is None or offset < total:
        response = fetch_page(url, offset=offset, start=start, end=end, length=page_size, extra_params=extra_params)
        body = response.json()["response"]

        if total is None:
            total = int(body["total"])
            logger.info(f"Total rows available: {total}")

        all_data.extend(body["data"])
        offset += page_size
        logger.info(f"Fetched {len(all_data)}/{total} rows so far")

    return all_data


END_DATE = date.today().isoformat()
START_DATE = (date.today() - timedelta(days=30)).isoformat()

In [0]:
consumption_data = fetch_all(CONSUMPTION_URL, start=START_DATE, end=END_DATE)
prices_data = fetch_all(PRICES_URL, start=START_DATE, end=END_DATE)

In [0]:
import json

def save_to_volume(data, file_name):
    file_path = volume_path + file_name
    with open(file_path, "w") as f:
        json.dump(data, f)
    logger.info(f"Saved {len(data)} records to {file_path}")


save_to_volume(consumption_data, "petroleum_raw.json")
save_to_volume(prices_data, "petroleum_prices_raw.json")